In [ ]:
import os, sys
from os import listdir
from os.path import join, basename, dirname
from datetime import datetime, timezone, timedelta
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from pytorch_metric_learning import losses
import numpy as np
import numpy.random as npr
from sklearn.model_selection import train_test_split

from shared import Data

device = torch.device('cuda')

def elapsed():
    return datetime.now(timezone(timedelta(hours=7))).strftime("%H:%M:%S")

# os.makedirs('/kaggle/working/cache', exist_ok=True)

In [ ]:
class Data1(Data):

    def __init__(
            self,
            sr: int = 32_000,
            duration: int = 5,      # секунды
            stride: float = 2.5,    # перекрытие в секундах
            num_segments: int = 5,
            is_train: bool = True
            ) -> None:
        super().__init__(sr, is_train)

        self._num_segments = num_segments

        # self._train_audio = r'/kaggle/input/competitions/birdclef-2026/train_audio'
        self._train_audio = r'data/train_audio'
        # self._cache_path = r'/kaggle/working/cache'
        self._cache_path = r'cache'
    
        self._segment_len = int(duration * sr)
        self._stride = int(stride * sr)

        clses = sorted(listdir(self._train_audio))
        for cls in clses:

            cls_path = join(self._train_audio, cls)
            cls_audios = listdir(cls_path)
            for audio in cls_audios:

                self._samples.append((
                    join(cls_path, audio),
                    self._cls2idx[cls]
                ))

        train_samples, valid_samples = train_test_split(
            self._samples,
            test_size=0.2,
            random_state=42
        )
        self._samples = train_samples if self._is_train else valid_samples


    def _make_slides(self, y):

        segments = []
        positions = list(range(0, max(1, len(y) - self._segment_len + 1), self._stride))

        if self._is_train:
            is_replace = len(positions) < self._num_segments
            positions = np.sort(npr.choice(positions, self._num_segments, replace=is_replace))
        else:
            if len(positions) > self._num_segments:
                idxs = np.linspace(0, len(positions)-1, self._num_segments).astype(int)
                positions = [positions[i] for i in idxs]
            else:
                positions = npr.choice( positions, self._num_segments, replace=True)

        for start in positions:
            segment = y[start:start+self._segment_len]

            if len(segment) < self._segment_len:
                pad = self._segment_len - len(segment)
                segment = torch.nn.functional.pad(segment, (0, pad))

            sps = self._get_sps(segment)
            segments.append(sps)

        return torch.stack(segments)

In [ ]:
batch_size = 8
train_dataset = DataLoader(
    Data1(),
    batch_size=batch_size, num_workers=4, shuffle=True, pin_memory=True, persistent_workers=True)
valid_dataset = DataLoader(
    Data1(is_train=False),
    batch_size=batch_size, num_workers=4, shuffle=False, pin_memory=True, persistent_workers=True)

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# потому что пока что только 2 спектограммы
old_conv = model.conv1
model.conv1 = nn.Conv2d(2, 64, 7, 2, 3, bias=False)

with torch.no_grad():
    model.conv1.weight[:] = old_conv.weight[:, :2]

model.fc = nn.Identity()
model.to(device);

In [ ]:
criterion = losses.ArcFaceLoss(
    num_classes=206,
    embedding_size=512,
).to(device)
optimizer = torch.optim.AdamW(
    params=list(model.parameters()) + list(criterion.parameters()),
    lr=1e-4     # лучше другой
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max', patience=1, factor=.5
)

In [ ]:
green = "\033[92m"
reset = "\033[0m"

EPOCH = 10
total_train, total_valid = len(train_dataset), len(valid_dataset)

best_acc = -float('inf')

# model.train()
for epoch in range(EPOCH):

    # train
    train_loss = 0.0
    for xb,yb in train_dataset:
        
        B, N, C, H, W = xb.shape
        x = xb.view(B*N, C, H, W).to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        y = yb.unsqueeze(1).repeat(1, N).view(-1)

        optimizer.zero_grad()

        emb = F.normalize(model(x))

        loss = criterion(emb, y)
        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= total_train

    # valid
    model.eval()
    valid_acc = 0.0
    with torch.no_grad():
        for xb,yb in valid_dataset:

            B, N, C, H, W = xb.shape
            x = xb.view(B*N, C, H, W).to(device, non_blocking=True)

            emb = F.normalize(model(x)).view(B, N, -1).mean(dim=1)
            W = F.normalize(criterion.W)

            similar = emb @ W

            pred = similar.argmax(dim=1)
            yb_cuda = yb.to(device, non_blocking=True)
            acc = (pred == yb_cuda).float().mean()

            valid_acc += acc.item()

    valid_acc /= total_valid

    print(f'Epoch: [ {epoch+1:^2} / {EPOCH} ]   lr: {optimizer.param_groups[0]["lr"]}   [{elapsed()}]')
    print(f'  acc: {green}{valid_acc:.4f}{reset},   TrainLoss: {train_loss:.4f}')

    if valid_acc > best_acc:
        best_acc = valid_acc

        torch.save({
            'model': model.state_dict(),
            'arcface': criterion.state_dict()
        # }, '/kaggle/working/best_params.pth')
        }, 'best_label.pth')
        print(f'  Модель {epoch+1} эпохи сохранена')

    model.train()
    scheduler.step(valid_acc)

In [ ]:
# torch.save({
#     'model': model.state_dict(),
#     'arcface': criterion.state_dict()
# # }, '/kaggle/working/best_params.pth')
# }, 'best_params.pth')

In [ ]:
# https://www.kaggle.com/code?import=true